# ADA v5 · Tensor trains, retrieval boundaries and masked denoising

This notebook exercises three experimental ADA v5 components using tiny
CPU fixtures. It reproduces two source-level problems and trains two
named research variants on independent synthetic tasks.

**Source.** Selected definitions are embedded from the ARcHi workspace's
Ada_v5/ada_v5.py, SHA-256 f1f0a54373fc8a906033097d374c7cb62b78a8104eb47a4c99fe32b495a386d2. The workspace architecture review
identifies unscoped retrieval and a diffusion input-width mismatch.
The fixtures below reproduce both with actual source definitions.

The TT projection is the historical implementation. Scoped retrieval
and masked denoising are notebook variants. This is a component study
with no claim to run the complete v5 stack, benchmark speed, or validate
its DeltaNet, spiking, predictive-coding, MTP or distributed paths.

### Run environment

Use Python 3.11 or 3.12 with Jupyter and a CPU PyTorch build (2.6 or later).
Select that environment as the kernel and choose **Restart Kernel and Run All**.
For a fresh environment, install Python packages with
**python -m pip install jupyterlab torch**, selecting the
[CPU wheels](https://pytorch.org/get-started/locally/) if desired.
The cells use synthetic tensors and make no network requests.

In [1]:
from io import BytesIO
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from types import SimpleNamespace

import torch
from torch import Tensor, nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(2026)
torch.use_deterministic_algorithms(True)
print("PyTorch", torch.__version__, "| device cpu | seed 2026")

ADAConfig = SimpleNamespace  # only the relevant config fields are needed here

PyTorch 2.11.0+cpu | device cpu | seed 2026


## Historical component definitions

These classes retain their original equations and state behavior.
The fixtures use smaller dimensions and call each component separately.

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        fp32 = x.float()
        norm = fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (fp32 * norm).to(x.dtype) * self.weight


def _closest_factor_pair(n: int) -> Tuple[int, int]:
    """Largest factor pair (a, b) with a <= b and a*b == n, a close to sqrt(n)."""
    a = int(math.isqrt(n))
    while a > 1 and n % a != 0:
        a -= 1
    return (a, n // a) if a > 1 else (1, n)


class TTLinear(nn.Module):
    """
    Two-core Tensor-Train factorized linear layer (MPO-style).
    W[i1,i2 ; j1,j2] = sum_r G1[i1,j1,r] * G2[r,i2,j2]

    Parameter count: in1*out1*rank + rank*in2*out2, vs dense in*out.
    Falls back to a dense nn.Linear automatically if in/out_features don't
    factorize into a useful pair (e.g. prime dimensions) — documented
    fallback, not a bug: TT-decomposition only pays off with room to split.
    """

    def __init__(self, in_features: int, out_features: int, rank: int, bias: bool = False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        in1, in2 = _closest_factor_pair(in_features)
        out1, out2 = _closest_factor_pair(out_features)

        self._dense_fallback = in1 == 1 or out1 == 1
        if self._dense_fallback:
            self.dense = nn.Linear(in_features, out_features, bias=bias)
            return

        self.in1, self.in2, self.out1, self.out2 = in1, in2, out1, out2
        self.rank = min(rank, in1 * out1, in2 * out2)

        self.core1 = nn.Parameter(torch.empty(in1, out1, self.rank))
        self.core2 = nn.Parameter(torch.empty(self.rank, in2, out2))
        nn.init.trunc_normal_(self.core1, std=(1.0 / math.sqrt(in1 * out1)) ** 0.5)
        nn.init.trunc_normal_(self.core2, std=(1.0 / math.sqrt(in2 * out2)) ** 0.5)

        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x: Tensor) -> Tensor:
        if self._dense_fallback:
            return self.dense(x)

        orig_shape = x.shape
        x = x.reshape(-1, self.in1, self.in2)  # (N, in1, in2)

        # step 1: contract in1 against core1 -> (N, out1, in2, rank)
        step1 = torch.einsum("nik,ior->nokr", x, self.core1.to(x.dtype))
        # step 2: contract in2/rank against core2 -> (N, out1, out2)
        step2 = torch.einsum("nokr,rkl->nol", step1, self.core2.to(x.dtype))

        out = step2.reshape(*orig_shape[:-1], self.out_features)
        if self.bias is not None:
            out = out + self.bias
        return out

    def compression_ratio(self) -> float:
        if self._dense_fallback:
            return 1.0
        tt_params = self.core1.numel() + self.core2.numel()
        dense_params = self.in_features * self.out_features
        return tt_params / dense_params


class DiffusionHead(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        self.steps = cfg.diffusion_steps
        self.vocab_size = cfg.vocab_size
        self.mask_token_id = cfg.vocab_size - 1

        self.diff_proj = nn.Linear(cfg.d_model, cfg.diffusion_hidden_dim)
        self.diff_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=cfg.diffusion_hidden_dim, nhead=8,
                dim_feedforward=cfg.diffusion_hidden_dim * 4,
                dropout=cfg.dropout, batch_first=True, norm_first=True,
            ) for _ in range(cfg.diffusion_num_layers)
        ])
        self.diff_norm = RMSNorm(cfg.diffusion_hidden_dim)
        self.diff_head = nn.Linear(cfg.diffusion_hidden_dim, cfg.vocab_size, bias=False)
        self.time_embed = nn.Embedding(cfg.diffusion_steps, cfg.diffusion_hidden_dim)

    def forward(self, x: Tensor, t: Optional[Tensor] = None) -> Tensor:
        B, T, _ = x.shape
        if t is None:
            t = torch.randint(0, self.steps, (B,), device=x.device)
        h = self.diff_proj(x) + self.time_embed(t).unsqueeze(1)
        for layer in self.diff_layers:
            h = layer(h)
        return self.diff_head(self.diff_norm(h))

    def diffuse_forward(self, target_ids: Tensor, x: Tensor) -> Tensor:
        B, T = target_ids.shape
        t = torch.randint(0, self.steps, (B,), device=target_ids.device)
        mask_prob = (t.float() / self.steps).unsqueeze(-1)
        mask = torch.rand(B, T, device=target_ids.device) < mask_prob
        logits = self.forward(x, t)
        loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), target_ids.reshape(-1), reduction="none")
        loss = loss.reshape(B, T)
        return (loss * mask).sum() / (mask.sum() + 1e-8)

    @torch.inference_mode()
    def generate_diffusion(self, x: Tensor, steps: int = None,
                            temperature: float = 1.0, top_k: int = 0) -> Tensor:
        steps = steps or self.steps
        B, T, _ = x.shape
        tokens = torch.full((B, T), self.mask_token_id, device=x.device, dtype=torch.long)
        for i in reversed(range(steps)):
            t = torch.full((B,), i, device=x.device, dtype=torch.long)
            logits = self.forward(x, t) / max(temperature, 1e-5)
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1)
                logits[logits < v[:, :, -1:]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            pred = torch.multinomial(probs.reshape(-1, probs.shape[-1]), 1).reshape(B, T)
            if i > 0:
                update_mask = torch.rand(B, T, device=x.device) < (i / steps)
                tokens[update_mask] = pred[update_mask]
            else:
                tokens = pred
        return tokens


class RetrievalModule(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        self.top_k = cfg.retrieval_top_k
        self.dim = cfg.retrieval_dim

        self.key_proj = nn.Linear(cfg.d_model, cfg.retrieval_dim, bias=False)
        self.value_proj = nn.Linear(cfg.d_model, cfg.retrieval_dim, bias=False)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=cfg.d_model, num_heads=cfg.retrieval_cross_attn_heads,
            kdim=cfg.retrieval_dim, vdim=cfg.retrieval_dim,
            batch_first=True, dropout=cfg.dropout,
        )

        self.register_buffer("key_cache", torch.empty(0, cfg.retrieval_dim))
        self.register_buffer("value_cache", torch.empty(0, cfg.retrieval_dim))
        self.register_buffer("token_cache", torch.empty(0, dtype=torch.long))
        self.max_cache_size = cfg.retrieval_max_cache_size

    def update_cache(self, hidden: Tensor, token_ids: Tensor):
        B, T, D = hidden.shape
        keys = self.key_proj(hidden.reshape(-1, D)).detach()
        values = self.value_proj(hidden.reshape(-1, D)).detach()
        tokens = token_ids.reshape(-1).detach()

        self.key_cache = torch.cat([self.key_cache, keys], dim=0)
        self.value_cache = torch.cat([self.value_cache, values], dim=0)
        self.token_cache = torch.cat([self.token_cache, tokens], dim=0)

        if self.key_cache.shape[0] > self.max_cache_size:
            self.key_cache = self.key_cache[-self.max_cache_size:]
            self.value_cache = self.value_cache[-self.max_cache_size:]
            self.token_cache = self.token_cache[-self.max_cache_size:]

    def retrieve(self, query: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        B, T, D = query.shape
        q_flat = self.key_proj(query.reshape(-1, D))

        if self.key_cache.shape[0] == 0:
            empty_k = torch.zeros(B * T, self.top_k, self.dim, device=query.device, dtype=query.dtype)
            empty_v = torch.zeros(B * T, self.top_k, self.dim, device=query.device, dtype=query.dtype)
            empty_t = torch.zeros(B * T, self.top_k, device=query.device, dtype=torch.long)
            return empty_k, empty_v, empty_t

        q_norm = F.normalize(q_flat, dim=-1)
        k_norm = F.normalize(self.key_cache, dim=-1)
        sim = q_norm @ k_norm.T

        k = min(self.top_k, self.key_cache.shape[0])
        topk_sim, topk_idx = torch.topk(sim, k, dim=-1)

        ret_keys = self.key_cache[topk_idx]
        ret_vals = self.value_cache[topk_idx]
        ret_toks = self.token_cache[topk_idx]

        if k < self.top_k:
            pad = self.top_k - k
            ret_keys = F.pad(ret_keys, (0, 0, 0, pad))
            ret_vals = F.pad(ret_vals, (0, 0, 0, pad))
            ret_toks = F.pad(ret_toks, (0, pad))

        return ret_keys, ret_vals, ret_toks

    def forward(self, x: Tensor) -> Tensor:
        B, T, D = x.shape
        ret_keys, ret_vals, _ = self.retrieve(x)
        ret_keys = ret_keys.reshape(B * T, self.top_k, self.dim)
        ret_vals = ret_vals.reshape(B * T, self.top_k, self.dim)
        q = x.reshape(B * T, 1, D)
        attn_out, _ = self.cross_attn(q, ret_keys, ret_vals)
        return attn_out.reshape(B, T, D)

## Tensor-train parity

A two-core tensor-train projection factors the weight across paired
input/output axes. Materialize the dense matrix independently and
compare both predictions and input gradients. Also check the historical
prime-dimension fallback. Fewer parameters is a property of this rank
and shape. Runtime would require a separate benchmark.

In [3]:
factorized = TTLinear(16, 16, rank=3, bias=True)
features = torch.randn(5, 16, requires_grad=True)
dense_weight = torch.einsum("ior,rkl->olik", factorized.core1, factorized.core2).reshape(16, 16)
factored_output = factorized(features)
reference_output = F.linear(features, dense_weight, factorized.bias)
torch.testing.assert_close(factored_output, reference_output, rtol=2e-6, atol=2e-6)
left = torch.autograd.grad(factored_output.square().sum(), features)[0]
right = torch.autograd.grad(reference_output.square().sum(), features)[0]
torch.testing.assert_close(left, right, rtol=3e-6, atol=3e-6)
prime = TTLinear(7, 11, rank=3)
assert prime._dense_fallback and prime(torch.zeros(2, 7)).shape == (2, 11)
print(f"TT forward/gradient parity: PASS | core parameter fraction: {factorized.compression_ratio():.3f}")
print("Prime-shape dense fallback: PASS")

TT forward/gradient parity: PASS | core parameter fraction: 0.375
Prime-shape dense fallback: PASS


## Historical retrieval and diffusion probes

Historical v5 appends a full sequence to its retrieval cache in training.
A later retrieval over the same sequence can therefore access suffix
positions. Evaluate two isolated snapshots of that state with an equal
two-token query and different suffix entries. Set top-k to include the
entire six-entry fixture to avoid an accidental passing negative control.

The historical loss call passes vocabulary-width logits to a diffusion
projection that accepts model-width features. Exercise that exact
argument shape and keep the resulting failure visible.

In [4]:
common = dict(d_model=16, dropout=0.0)
retrieval_cfg = ADAConfig(
    **common, retrieval_dim=8, retrieval_top_k=6, retrieval_cross_attn_heads=2,
    retrieval_max_cache_size=32,
)
source_retriever = RetrievalModule(retrieval_cfg).eval()
source_shadow = RetrievalModule(retrieval_cfg).eval()
source_shadow.load_state_dict(source_retriever.state_dict())
fixed_prefix = torch.randn(1, 2, 16)
corpus_a = torch.cat((fixed_prefix, torch.randn(1, 4, 16)), dim=1)
corpus_b = torch.cat((fixed_prefix, 4 * torch.randn(1, 4, 16)), dim=1)
with torch.no_grad():
    source_retriever.update_cache(corpus_a, torch.arange(6).view(1, 6))
    source_shadow.update_cache(corpus_b, torch.arange(6).view(1, 6))
    _, _, candidate_ids = source_retriever.retrieve(fixed_prefix)
    assert bool((candidate_ids >= 2).any())
    delta = (source_retriever(fixed_prefix) - source_shadow(fixed_prefix)).abs().max().item()
assert delta > 1e-5
print(f"Historical same-query/suffix-cache delta: {delta:.6f} | FAIL (expected)")

denoise_cfg = ADAConfig(
    **common, vocab_size=17, diffusion_hidden_dim=16, diffusion_steps=4,
    diffusion_num_layers=1,
)
diffusion = DiffusionHead(denoise_cfg)
target_ids = torch.randint(16, (2, 6))
vocabulary_logits = torch.zeros(2, 6, denoise_cfg.vocab_size)
try:
    diffusion.diffuse_forward(target_ids, vocabulary_logits)
except RuntimeError as error:
    if "mat1 and mat2 shapes cannot be multiplied" not in str(error):
        raise
    print("Historical vocabulary-to-model width probe: FAIL (expected)", str(error))
else:
    raise AssertionError("historical mismatch was not reproduced")

Historical same-query/suffix-cache delta: 1.413150 | FAIL (expected)
Historical vocabulary-to-model width probe: FAIL (expected) mat1 and mat2 shapes cannot be multiplied (12x17 and 16x16)


## Sequence-scoped autoregressive retrieval

This reference retains cosine top-k retrieval and cross-attention.
Each batch row has its own candidate stream. At position t, candidates
are drawn only from earlier positions in a fixed recent window. There is
no mutable history shared across rows or forward calls. The Python loop
serves as a simple reference for a future optimized implementation.

In [5]:
class ScopedRetrieval(nn.Module):
    def __init__(self, config, window=5):
        super().__init__()
        if window < 1 or config.retrieval_top_k < 1:
            raise ValueError("window and top_k must be positive")
        self.window = window
        self.top_k = config.retrieval_top_k
        width, key_width = config.d_model, config.retrieval_dim
        self.keys = nn.Linear(width, key_width, bias=False)
        self.values = nn.Linear(width, key_width, bias=False)
        self.attention = nn.MultiheadAttention(
            width, config.retrieval_cross_attn_heads, kdim=key_width,
            vdim=key_width, batch_first=True,
        )

    def forward(self, x):
        keys = self.keys(x)
        normalized = F.normalize(keys, dim=-1)
        values = self.values(x)
        outputs = [torch.zeros_like(x[:, :1])]
        for t in range(1, x.shape[1]):
            start = max(0, t - self.window)
            scores = (
                normalized[:, start:t] * normalized[:, t:t + 1]
            ).sum(-1)
            indices = scores.topk(min(self.top_k, t - start), dim=1).indices + start
            index = indices.unsqueeze(-1).expand(-1, -1, keys.shape[-1])
            fetched_keys = keys.gather(1, index)
            fetched_values = values.gather(1, index)
            read, _ = self.attention(x[:, t:t + 1], fetched_keys, fetched_values, need_weights=False)
            outputs.append(read)
        return torch.cat(outputs, dim=1)


class ToyV5LM(nn.Module):
    def __init__(self, config, vocab_size=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, config.d_model)
        self.tt = TTLinear(config.d_model, config.d_model, rank=3)
        self.retrieval = ScopedRetrieval(config)
        self.norm = RMSNorm(config.d_model)
        self.head = nn.Linear(config.d_model, vocab_size)

    def forward(self, ids):
        x = self.embed(ids)
        x = x + F.silu(self.tt(x))
        return self.head(self.norm(x + self.retrieval(x)))


model = ToyV5LM(retrieval_cfg)
ids = torch.tensor([[1, 2, 3, 4, 5, 6]])
altered = torch.tensor([[1, 2, 3, 4, 10, 11]])


@torch.no_grad()
def assert_retrieval_isolation(model):
    model.eval()
    baseline = model(ids)
    torch.testing.assert_close(baseline[:, :4], model(altered)[:, :4], rtol=0, atol=2e-6)
    torch.testing.assert_close(baseline[:, :4], model(ids[:, :4]), rtol=0, atol=2e-6)
    torch.testing.assert_close(baseline, model(torch.cat((ids, altered)))[:1], rtol=0, atol=2e-6)
    print("Prefix, truncation and batch-row isolation: PASS")


assert_retrieval_isolation(model)

Prefix, truncation and batch-row isolation: PASS


## Autoregressive optimization and evaluation

Both partitions follow modular addition, with disjoint start IDs.
The exercise verifies gradient flow into TT and retrieval projections
and reruns the prefix and batch isolation checks after optimization.

In [6]:
def cyclic_examples(starts, length, vocab):
    return (torch.as_tensor(starts)[:, None] + torch.arange(length)[None, :]) % vocab


def next_token_metrics(logits, ids):
    prediction = logits[:, :-1]
    labels = ids[:, 1:]
    loss = F.cross_entropy(prediction.flatten(0, 1), labels.flatten())
    accuracy = (prediction.argmax(-1) == labels).float().mean()
    return loss, accuracy

In [7]:
train_ids = cyclic_examples(range(12), 10, 16)
val_ids = cyclic_examples(range(12, 16), 10, 16)
assert torch.unique(torch.cat((train_ids, val_ids)), dim=0).shape[0] == 16


@torch.no_grad()
def evaluate_lm():
    model.eval()
    return [value.item() for value in next_token_metrics(model(val_ids), val_ids)]


before, _ = evaluate_lm()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.018)
model.train()
for step in range(32):
    optimizer.zero_grad(set_to_none=True)
    loss, _ = next_token_metrics(model(train_ids), train_ids)
    assert torch.isfinite(loss)
    loss.backward()
    if step == 0:
        assert model.tt.core1.grad.abs().sum() > 0
        assert model.retrieval.values.weight.grad.abs().sum() > 0
    optimizer.step()
after, accuracy = evaluate_lm()
assert after < before
print(f"Scoped retrieval held-out CE: {before:.6f} -> {after:.6f}; accuracy {accuracy:.1%}")
assert_retrieval_isolation(model)

with torch.no_grad():
    generated = torch.tensor([[13, 14]])
    for _ in range(6):
        last = model(generated)[:, -1].argmax(-1, keepdim=True)
        generated = torch.cat((generated, last), dim=1)
assert generated.shape == (1, 8)
print("Greedy IDs:", generated.tolist())

Scoped retrieval held-out CE: 2.724228 -> 0.131861; accuracy 94.4%
Prefix, truncation and batch-row isolation: PASS
Greedy IDs: [[13, 14, 15, 0, 1, 2, 3, 4]]


## Separate bidirectional masked-token denoising

Reuse the original time-conditioned output head, but embed explicitly
corrupted token IDs into model-width features before passing them in.
Compute loss only at masked positions. This is a masked-token
denoising demonstration. A discrete diffusion process would require
a defined corruption transition, reverse update and independently tested
sampling. This bidirectional path stays outside autoregressive scoring.

In [8]:
class MaskedDenoiser(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed = nn.Embedding(config.vocab_size, config.d_model)
        self.head = DiffusionHead(config)

    def forward(self, clean_ids, mask, diffusion_step):
        corrupted = clean_ids.masked_fill(mask, self.head.mask_token_id)
        return self.head(self.embed(corrupted), diffusion_step)


torch.manual_seed(2027)
denoiser = MaskedDenoiser(denoise_cfg)
denoise_opt = torch.optim.AdamW(denoiser.parameters(), lr=0.015)


def masked_objective(ids, offset):
    mask = ((torch.arange(ids.shape[1]) + offset) % 3 == 1).expand_as(ids)
    t = torch.full((ids.shape[0],), 2, dtype=torch.long)
    logits = denoiser(ids, mask, t)
    return F.cross_entropy(logits[mask], ids[mask]), logits[mask].argmax(-1), ids[mask]


denoiser.eval()
with torch.no_grad():
    denoise_before = masked_objective(val_ids, 0)[0].item()
denoiser.train()
for step in range(48):
    denoise_opt.zero_grad(set_to_none=True)
    denoise_loss, _, _ = masked_objective(train_ids, step % 3)
    assert torch.isfinite(denoise_loss)
    denoise_loss.backward()
    denoise_opt.step()
denoiser.eval()
with torch.no_grad():
    denoise_after, predicted, truth = masked_objective(val_ids, 0)
assert denoise_after.item() < denoise_before
print(f"Masked-token held-out CE: {denoise_before:.6f} -> {denoise_after.item():.6f}")
print(f"Held-out masked accuracy: {(predicted == truth).float().mean().item():.1%}")

Masked-token held-out CE: 3.022021 -> 2.792559
Held-out masked accuracy: 25.0%


## Decision

Preserve the failing historical controls for regression work. The
tensor train and sequence-boundary fixtures are ready for larger parity
suites. Before introducing a persistent retrieval cache, define
session ownership, time and length bounds, resets and leakage checks.
Compare against matched dense and no-retrieval baselines before drawing
performance or capability conclusions.